# v3 - the "true accuracy" study: START HERE

## Why v3 exists
In v1/v2 some models scored almost 100 %. That is not believable, and the audit in this study shows why: **in the public data every file holds one class only**, so a program that just recognises *which dataset a text came from* scores about **99.6 %** without understanding any scam. v3 is built so that the score you read cannot be explained that way.

## What v3 does differently
| Problem | Fix in v3 |
|---|---|
| Score = "which dataset is this?" | Two regimes are scored separately. **public** = public data (optimistic, style reveals the class). **hard** = a style-matched set written for this project: polite scams with no keywords and safe messages full of scam words, Puerto Rico flavoured, EN + ES. Both classes are written alike, so only the *content* can give the answer. The audit proves metadata alone scores at chance there. |
| Aligned pairs let one channel do all the work | **Mixed pairs under the proposal's TOAD rule**: a case is fraud *only if the call AND the message are both fraud*. Fraud call + safe message = NOT fraud (and vice versa). A perfect single-channel model tops out at ~85 %; only using both channels can reach 100 %. |
| Copies of the same script on both sides of a split | Near-identical scripts are clustered into **families**; a whole family sits in ONE bucket (train / validation / test fold). Checked: 0 shared families, 0 shared ids. |
| Wrong labels | A **label audit** lists the items whose label a held-out model contradicts most confidently, for you to read (`outputs/v3/tables/label_audit_suspects_to_review.csv`). |
| Is something leaking? | **Label-shuffle check**: models trained on shuffled labels must fall to the majority-class rate (60 %). **Shortcut baselines** and **ceilings** are printed next to every result. |
| Over-confident error bars | Intervals resample whole script families (cluster bootstrap), not single records. |
| One lucky split | Family-safe cross-validation over 4 folds for development; the test buckets are used **once**. |

## Honest limits
* The hard set is **written by me (an AI), not recorded from real scams**. Labels are correct by construction, but real DACO cases will differ. Put real cases in `data/external/pr_cases.csv` (columns in its header) to test on reality.
* The hard regime has few script families (test: 6 call families, 7 message families), so its intervals are wide. That width is the honest uncertainty.
* Under the TOAD rule single-channel models are capped near 85 %, so combined > single is partly the *design* of the task; the notebooks show per-pair-type accuracy so you can see exactly where each model fails.

## Run order (VS Code: open the folder `ToadWatch_PR`, pick the Python kernel, *Run All* each)
| # | Notebook | Does | Time |
|---|---|---|---|
| 1 | `v3_01_prepare_data.ipynb` | clean, families, buckets, mixed pairs, lock the test buckets | ~3 min |
| 2 | `v3_02_audit_data.ipynb` | leakage checks, ceilings, shortcut baselines, label audit | ~1 min |
| 3 | `v3_03_train_tune_cv.ipynb` | tune, family-safe CV of 7 models, freeze settings | ~25 min |
| 4 | `v3_04_sanity_shuffle.ipynb` | shuffled-label check (must collapse to chance) | ~10 min |
| 5 | `v3_05_final_test_once.ipynb` | **one-shot** test, true accuracy with intervals | ~3 min |
| 6 | `v3_06_charts.ipynb` | every chart (saved to `outputs/v3/figures`) | ~1 min |

The last cell of notebooks 5 and 6 regenerates the **Word report + PDF and the PowerPoint + PDF** (folder `report/v3/`) from the run you just made.

Terminal: `python scripts/run_v3.py all` (steps 1-4, charts, report) then `python scripts/run_v3.py final`.

In [1]:
import sys, importlib
from pathlib import Path
print("python", sys.version.split()[0])
for m in ["pandas","numpy","sklearn","scipy","matplotlib","seaborn","pyarrow","joblib"]:
    try: print(f"  ok   {m} {importlib.import_module(m).__version__}")
    except Exception as e: print(f"  MISSING {m}  ->  pip install {m}")
root = Path.cwd().parent
print("notebooks folder ok:", Path.cwd().name == "notebooks")
for d in ["Call Safe data","Call Dangerous data","SMS, Email, URL Safe","SMS, Email, URL Dangerous"]:
    print(("  ok   " if (root.parent / d).exists() else "  MISSING ") + d)

python 3.14.6


  ok   pandas 3.0.3
  ok   numpy 2.4.6


  ok   sklearn 1.9.0
  ok   scipy 1.18.0
  ok   matplotlib 3.11.0


  ok   seaborn 0.13.2
  ok   pyarrow 23.0.1
  ok   joblib 1.5.3
notebooks folder ok: True
  ok   Call Safe data
  ok   Call Dangerous data
  ok   SMS, Email, URL Safe
  ok   SMS, Email, URL Dangerous
